In [14]:
import pandas as pd
import numpy as np
from statsmodels.tsa.stattools import adfuller, kpss

%store -r df
adf_result = adfuller(df['CLOSE'].dropna(), regression='c', result_object=True)
kpss_result = kpss(df['CLOSE'].dropna(), regression='c', result_object=True )
print('--------- ADF - test  -------------- ')
print(adf_result)
print('--------- KPSS - test  -------------- ')
print(kpss_result)

--------- ADF - test  -------------- 
ADFullerResult
ADF Statistic: -2.29422
P-value: 0.17384
Used Lag: 18
Nobs: 738
Critical Values: {'1%': np.float64(-3.4392418578608983), '5%': np.float64(-2.86546426916434), '10%': np.float64(-2.568859710342903)}

--------- KPSS - test  -------------- 
KPSSResult
KPSS Statistic: 0.66871
P-value: 0.01639
Lags: 17
Critical Values: {'10%': 0.347, '5%': 0.463, '2.5%': 0.574, '1%': 0.739}



In [15]:
# функции для работы с сырыми массивами (raw=True)
def get_adf_p(x):
    if np.all(x == x[0]):
        return np.nan
    try:
        return adfuller(x, regression="c", autolag="AIC", result_object=True).pvalue
    except:
        return np.nan

def get_kpss_p(x):
    if np.all(x == x[0]):
        return np.nan
    try:
        return kpss(x, regression="c", nlags="auto", result_object=True).pvalue
    except:
        return np.nan

def stationarity_rolling_window(df, column_name, min_pct=0.20, max_pct=0.70, step_pct=0.05, alpha_adf=0.05, alpha_kpss=0.05):
    series = df[column_name].dropna()
    total_len = len(series)

    min_window = int(total_len * min_pct)
    max_window = int(total_len * max_pct)
    step = max(1, int(total_len * step_pct))
    window_sizes = range(min_window, max_window + 1, step)

    print(f"Анализ ряда длиной {total_len}")

    successful_windows = [] # для сводной таблицы по p-value тестов

    for window_size in window_sizes:
        adf_series = series.rolling(window=window_size).apply(get_adf_p, raw=True)
        kpss_series = series.rolling(window=window_size).apply(get_kpss_p, raw=True)

        # иаска для выполнений условий стационарности, применяемая к рядам
        valid_mask = (adf_series < alpha_adf) & (kpss_series >= alpha_kpss)

        # Сохраняем столбец ADF в итоговый датафрейм (только валидные значения, остальные NaN)
        df[f'ADF {window_size}'] = adf_series.where(valid_mask)
        df[f'KPSS {window_size}'] = kpss_series.where(valid_mask)

        # информация для таблицы results_df только по успешным точкам
        end_indices = series.index[valid_mask]
        for end_label in end_indices:
            # найти порядковый номер конечной точки в исходном ряду
            end_idx = series.index.get_loc(end_label)
            start_idx = end_idx - window_size + 1
            start_label = series.index[start_idx]

            successful_windows.append({
                "window_size": window_size,
                "window_pct": round(window_size / total_len, 2),
                "start_index": start_label,
                "end_index": end_label,
                "start_datetime": df.loc[start_label, "DATETIME"] if "DATETIME" in df.columns else start_label,
                "end_datetime": df.loc[end_label, "DATETIME"] if "DATETIME" in df.columns else end_label,
                "adf_p_value": adf_series.loc[end_label],
                "kpss_p_value": kpss_series.loc[end_label] })

    results_df = pd.DataFrame(successful_windows)

    # удаляем колонки ADF и KPSS со значениями nan, оставляем только те где выполняются словия и по ADF и по KPSS
    df = df.loc[:, ~(df.columns.str.startswith('ADF ') | df.columns.str.startswith('KPSS ')) | df.notna().any()]
    return results_df, df

In [29]:
r_window_min_pct = float(input(f'Enter MINIMAL % for rolling window in decimal format:'))
r_window_max_pct = float(input(f'Enter MAXIMUM % for rolling window in decimal format:'))
r_window_step_pct = float(input(f'Enter STEP % for rolling window in decimal format:'))
adf_p_user = float(input(f'Enter ADF p-value in decimal format:'))
kpss_p_user =float(input(f'Enter KPSS p-value in decimal format:'))
print(f'''Parameters:
        min rolling window: {r_window_min_pct * 100}%
        max rolling window: {r_window_max_pct* 100}%
        window step:        {r_window_step_pct * 100}%
        ADF p-value used:   {adf_p_user}
        KPSS p-value used:  {adf_p_user}''')

Parameters:
        min rolling window: 30.0%
        max rolling window: 70.0%
        window step:        3.0%
        ADF p-value used:   0.1
        KPSS p-value used:  0.1


In [30]:
import warnings
from statsmodels.tools.sm_exceptions import (InterpolationWarning, ValueWarning)
warnings.filterwarnings('ignore', category=InterpolationWarning)
warnings.filterwarnings('ignore', category=ValueWarning)

pvalue_res, df_aug = stationarity_rolling_window(df=df, column_name="CLOSE", min_pct=r_window_min_pct, max_pct=r_window_max_pct, step_pct=r_window_step_pct, alpha_adf=adf_p_user, alpha_kpss=kpss_p_user)
df_aug.head()

Анализ ряда длиной 757


,TICKER,DATETIME,CLOSE,h-cleaned,ADF 227,KPSS 227,ADF 264,KPSS 264,ADF 301,KPSS 301,ADF 523,KPSS 523,ADF 249,KPSS 249,ADF 271,KPSS 271,ADF 293,KPSS 293,ADF 315,KPSS 315
0,SVZ6SVH7 [FUTSPREAD],2026-09-01 07:00:00,1.71,1.75,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,SVZ6SVH7 [FUTSPREAD],2026-09-01 08:00:00,1.69,1.75,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,SVZ6SVH7 [FUTSPREAD],2026-09-01 09:00:00,1.69,1.75,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,SVZ6SVH7 [FUTSPREAD],2026-09-01 10:00:00,1.72,1.75,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,SVZ6SVH7 [FUTSPREAD],2026-09-01 11:00:00,1.75,1.75,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
752,SVZ6SVH7 [FUTSPREAD],2026-10-27 13:00:00,1.56,1.56,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
753,SVZ6SVH7 [FUTSPREAD],2026-10-27 14:00:00,1.54,1.56,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
754,SVZ6SVH7 [FUTSPREAD],2026-10-27 15:00:00,1.54,1.56,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
755,SVZ6SVH7 [FUTSPREAD],2026-10-27 16:00:00,1.52,1.56,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [31]:
nedf=df_aug.dropna(subset=df_aug.columns[4:], how='all')
nedf

,TICKER,DATETIME,CLOSE,h-cleaned,ADF 227,KPSS 227,ADF 264,KPSS 264,ADF 301,KPSS 301,ADF 523,KPSS 523,ADF 249,KPSS 249,ADF 271,KPSS 271,ADF 293,KPSS 293,ADF 315,KPSS 315
538,SVZ6SVH7 [FUTSPREAD],2026-10-11 12:00:00,1.51,1.51,0.075538,0.1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
539,SVZ6SVH7 [FUTSPREAD],2026-10-11 14:00:00,1.51,1.51,0.085197,0.1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
541,SVZ6SVH7 [FUTSPREAD],2026-10-11 16:00:00,1.51,1.51,0.087936,0.1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
545,SVZ6SVH7 [FUTSPREAD],2026-10-12 08:00:00,1.51,1.51,0.086533,0.1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
546,SVZ6SVH7 [FUTSPREAD],2026-10-12 09:00:00,1.51,1.51,0.092187,0.1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
547,SVZ6SVH7 [FUTSPREAD],2026-10-12 10:00:00,1.50,1.50,0.094642,0.1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
558,SVZ6SVH7 [FUTSPREAD],2026-10-12 21:00:00,1.49,1.49,NaN,NaN,0.017509,0.050711,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
559,SVZ6SVH7 [FUTSPREAD],2026-10-12 22:00:00,1.48,1.48,NaN,NaN,0.031801,0.057913,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
560,SVZ6SVH7 [FUTSPREAD],2026-10-12 23:00:00,1.48,1.48,NaN,NaN,0.024544,0.065076,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
586,SVZ6SVH7 [FUTSPREAD],2026-10-14 14:00:00,1.57,1.57,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.089188,0.1,NaN,NaN,NaN,NaN


In [32]:
import re
import plotly.graph_objects as go
from plotly.subplots import make_subplots

fig = make_subplots(specs=[[{"secondary_y": True}]])
fig.add_trace(go.Scatter(x=df_aug['DATETIME'], y=df_aug['h-cleaned'], name=df_aug['TICKER'].iloc[1]), secondary_y=False)
# числовые суффиксы ADF
suffixes = sorted({match.group(1) for col in df_aug.columns if (match := re.fullmatch(r'ADF\s+(\d+)', col))}, key=int)  # подсказал ИИ

for suffix in suffixes:
    adf_col = f'ADF {suffix}'
    kpss_col = f'KPSS {suffix}'

    if kpss_col not in df_aug.columns:
        continue

    for col in [adf_col, kpss_col]:
        fig.add_trace(go.Scatter(x=df_aug['DATETIME'], y=df_aug[col], name=f'{col} rolling'), secondary_y=True)

fig.show()

In [19]:
def adf_rolling(x):
    return adfuller(x, regression='c', autolag="AIC", result_object=True).pvalue
def kpss_rolling(x):
    return kpss(x, regression='c', nlags='auto', result_object=True).pvalue
# window = 300
# df['ADF roll'] = df['<CLOSE>'].rolling(window).apply(adf_rolling, raw=True)
# df['KPSS roll'] = df['<CLOSE>'].rolling(window).apply(kpss_rolling, raw=True)
# df.iloc[:61]

In [20]:
def stationarity_results_to_df (df, min_pct=0.20, max_pct=0.70, step_pct=0.05):
    total_len = len(df['CLOSE'])
    min_window = int(min_pct * total_len)
    max_window = int(max_pct * total_len)
    step_window = max(1, int(step_pct * total_len))
    window_sizes = range(min_window, max_window + 1, step_window)

    for window in window_sizes:
        df[f'ADF {window}'] = df['CLOSE'].rolling(window).apply(adf_rolling, raw=True)
        df[f'KPSS {window}'] = df['CLOSE'].rolling(window).apply(kpss_rolling, raw=True)

    return df

In [21]:
min_pct = float(input('enter min % in decimal format'))
max_pct = float(input('enter max bond for percentage in decimal format'))
step_pct = float(input('enter step % in decimal format'))

new_df = stationarity_results_to_df(df, min_pct, max_pct, step_pct)

C:\Users\user\AppData\Local\Temp\ipykernel_6972\3036405258.py:4: InterpolationWarning: The test statistic is outside of the range of p-values available in the look-up table. The actual p-value is smaller than the p-value returned.
  return kpss(x, regression='c', nlags='auto', result_object=True).pvalue
C:\Users\user\AppData\Local\Temp\ipykernel_6972\3036405258.py:4: InterpolationWarning: The test statistic is outside of the range of p-values available in the look-up table. The actual p-value is smaller than the p-value returned.
  return kpss(x, regression='c', nlags='auto', result_object=True).pvalue
C:\Users\user\AppData\Local\Temp\ipykernel_6972\3036405258.py:4: InterpolationWarning: The test statistic is outside of the range of p-values available in the look-up table. The actual p-value is smaller than the p-value returned.
  return kpss(x, regression='c', nlags='auto', result_object=True).pvalue
C:\Users\user\AppData\Local\Temp\ipykernel_6972\3036405258.py:4: InterpolationWarning